# 🎵 ACE-Step 1.5 XL｜Colab 歌曲生成 + LRC（v1）

在 Colab 上執行 ACE Studio × StepFun 的開源歌曲模型 **ACE-Step 1.5**，使用最高品質的 **XL（4B DiT）+ 4B LM** 組合，附批量生成、Seed 管理、LRC 歌詞時間軸、MP3 輸出，以及 Google Drive 模型備份。

## 使用方式

1. 「檔案 → 在雲端硬碟中儲存複本」，之後都從自己的副本開啟
2. 「執行階段 → 變更執行階段類型」選 **A100**（XL + 4B LM 官方建議 24GB 以上）
3. 修改 **③ 歌曲設定**，在 **③-2** 貼上 CAPTION 和 LYRICS
4. 有需要時才調整 **③-3 進階設定**，或在 **③-4** 指定 BPM／調性／拍號
5. 「執行階段 → 全部執行」
6. 想用**圖形介面**：保持 ⑤G 的 `USE_GUI` 勾選，「全部執行」後到 ⑤G 的輸出找網址和帳號密碼；此時 ⑤ 表單批量會自動略過。想用原本的表單流程就取消勾選。

- 同一個 Runtime 換歌：改完 ③／③-2／③-3／③-4 後直接執行 ⑤；④ 偵測到模型已載入會自動略過。
- 改了 ③ 的 `DIT_MODEL` 要重新執行 ④，會自動換模型。
- ⑤B、⑦ 預設不執行，需要時勾選後再單獨執行。

## 寫法重點

| 項目 | 說明 |
|---|---|
| CAPTION | 曲風、樂器、人聲、情緒、速度感等，英文效果最穩，可以寫一段描述 |
| LYRICS | `[Verse]` `[Chorus]` 等段落標籤可以保留；留空或勾 `INSTRUMENTAL` 會生成純音樂 |
| 長度 | `DURATION_SEC` 為 `-1` 時由 LM 依歌詞決定；也可以指定 10～600 秒 |
| 曲式 | ③-4 的 BPM、調性、拍號留空就交給 LM 決定；填了就照填的生成。圖形介面可以先按「先讓 LM 規劃曲式」看結果，改好再生成 |
| Thinking | LM 先規劃曲式與音樂結構再交給 DiT（官方預設開啟），關掉會比較快但結構比較鬆 |
| LRC | `acestep`：用 ACE-Step 生成時的注意力對歌詞，不用另外安裝、幾乎不花時間（預設）｜`whisper`：和 YuE2 版相同的 Demucs 分離人聲 + Whisper 對歌詞，第一次用到會自動安裝。⑤B 補做 LRC 一律用 whisper。 |
| 步數 | xl-sft 預設 50 步、xl-turbo 預設 8 步；③-3 的 `INFERENCE_STEPS` 為 0 時用預設值 |

## 運作方式

ACE-Step 需要 Python 3.11 和自己固定版本的 PyTorch，和 Colab 內建環境不相容，所以裝在獨立的 venv 裡，由背景程序載入模型；notebook 和圖形介面只負責送出工作、整理檔案和 LRC。

## 模型備份

勾選 ① 的 `USE_DRIVE_BACKUP` 後，最後一格 ⑧ 會在模型第一次載入成功後，把程式碼和模型備份到 Google Drive（XL + 4B LM 加上主模型，合計數十 GB，請確認雲端硬碟空間）。
平常仍從 Hugging Face／GitHub 下載（通常比 Drive 快）；只有它們無法使用時，才會自動改用 Drive 備份。

## 授權與致謝

- 本 notebook 的程式碼以 MIT License 釋出。
- 執行時會從官方來源安裝 [ACE-Step 1.5](https://github.com/ace-step/ACE-Step-1.5) 與（選用 whisper LRC 時）[lyric-align](https://github.com/ijuinryukichi/lyric-align)（MIT），本 notebook 不包含它們的程式碼。
- 模型權重（ACE-Step、Whisper 等）各有自己的授權條款，使用前請自行確認，尤其是商業用途。

In [ ]:
# @title ① 🔧 環境檢查與安裝
# ACE-Step 官方用 uv 安裝在獨立的 Python 3.11 venv，不會動到 Colab 內建的套件，所以不需要重新啟動執行階段。
# 同一個 Runtime 已經裝好時直接略過；只會重新做一次 import 測試。

import os
import sys
import shutil
import hashlib
import subprocess
from pathlib import Path

# @markdown ### 💾 Google Drive 備份
USE_DRIVE_BACKUP = True  # @param {type:"boolean"}
BACKUP_ROOT = "/content/drive/MyDrive/AI模型備份/ACE-Step"  # @param {type:"string"}

# @markdown ### 進階（平常不用勾）
FORCE_UPDATE_ACESTEP = False  # @param {type:"boolean"}

ACE_REPO = Path("/content/ACE-Step-1.5")
ACE_REPO_URL = "https://github.com/ace-step/ACE-Step-1.5.git"
VENV_PY = ACE_REPO / ".venv" / "bin" / "python"
SYNC_MARK = ACE_REPO / ".colab_sync_ok"
ACE_RESTORED = set()


def run_cmd(cmd, check=True, cwd=None):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run([str(c) for c in cmd], check=check, cwd=cwd)


def venv_env():
    """給 ACE-Step venv 用的環境變數：Colab 的設定有些會讓 venv 裡的套件出錯。"""
    env = os.environ.copy()
    env["MPLBACKEND"] = "Agg"        # Colab 的 matplotlib inline 後端在 venv 裡不存在
    env.pop("PYTHONPATH", None)       # 避免混到 Colab 內建 Python 的套件
    env["LOGURU_LEVEL"] = "INFO"      # 隱藏 DEBUG 訊息
    env["PYTHONUNBUFFERED"] = "1"
    return env


def lock_hash():
    lock = ACE_REPO / "uv.lock"
    return hashlib.md5(lock.read_bytes()).hexdigest() if lock.exists() else "none"


def fresh_ace_import_test():
    # 用全新的 process 在 venv 裡測試
    code = ("import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())\n"
            "import acestep.handler, acestep.llm_inference, acestep.inference\n"
            "print('ACE-Step import OK')\n")
    if not VENV_PY.exists():
        return False, "venv 不存在"
    r = subprocess.run([str(VENV_PY), "-c", code], cwd=str(ACE_REPO), env=venv_env(),
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0, r.stdout


# ---------- Google Drive 備份工具 ----------
import tarfile
from datetime import datetime

DRIVE_MYDRIVE = Path("/content/drive/MyDrive")
BACKUP_DIR = Path(BACKUP_ROOT)

if USE_DRIVE_BACKUP and not DRIVE_MYDRIVE.exists():
    from google.colab import drive
    drive.mount("/content/drive")


def drive_ready() -> bool:
    return USE_DRIVE_BACKUP and DRIVE_MYDRIVE.exists()


def backup_done(name: str) -> bool:
    return drive_ready() and (BACKUP_DIR / name / ".complete").exists()


def dir_size_gb(p: Path) -> float:
    return sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file()) / 1024**3


def _begin(name: str) -> Path:
    tmp = BACKUP_DIR / f"{name}.partial"
    shutil.rmtree(tmp, ignore_errors=True)
    tmp.mkdir(parents=True)
    return tmp


def _finish(tmp: Path, name: str):
    final = BACKUP_DIR / name
    shutil.rmtree(final, ignore_errors=True)
    tmp.rename(final)
    (final / ".complete").write_text(datetime.now().isoformat(timespec="seconds"))


def backup_dirs(name: str, sources: dict):
    """把多個資料夾備份到 BACKUP_DIR/name/<子資料夾>。中途失敗不會留下「看起來完整」的備份。"""
    tmp = _begin(name)
    for sub, src in sources.items():
        print(f"   複製 {sub}（{dir_size_gb(src):.1f} GB）……")
        shutil.copytree(src, tmp / sub, symlinks=False)
    _finish(tmp, name)


def backup_code_tar(name: str, src: Path, exclude=()):
    """程式碼打包成 code.tar.gz（排除大型資料夾）。"""
    tmp = _begin(name)
    exclude = set(exclude) | {".git"}

    def keep(ti):
        parts = Path(ti.name).parts
        return None if len(parts) > 1 and parts[1] in exclude else ti

    with tarfile.open(tmp / "code.tar.gz", "w:gz") as tf:
        tf.add(src, arcname="code", filter=keep)
    _finish(tmp, name)


def restore_dirs(name: str, dst_parent: Path, only=None) -> bool:
    """從 Drive 把備份複製回本機（不要直接從 Drive 載入模型，會很慢）。"""
    if not backup_done(name):
        return False
    for sub in sorted((BACKUP_DIR / name).iterdir()):
        if not sub.is_dir() or (only and sub.name not in only):
            continue
        dst = Path(dst_parent) / sub.name
        print(f"   從 Drive 還原 {sub.name}（{dir_size_gb(sub):.1f} GB）……")
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(sub, dst)
    return True


def restore_code_tar(name: str, dst: Path) -> bool:
    if not backup_done(name):
        return False
    tmp = Path("/content/_code_restore")
    shutil.rmtree(tmp, ignore_errors=True)
    with tarfile.open(BACKUP_DIR / name / "code.tar.gz") as tf:
        try:
            tf.extractall(tmp, filter="data")
        except TypeError:
            tf.extractall(tmp)
    dst.mkdir(parents=True, exist_ok=True)
    for p in (tmp / "code").iterdir():
        target = dst / p.name
        if target.is_dir():
            shutil.rmtree(target)
        elif target.exists():
            target.unlink()
        shutil.move(str(p), str(target))
    shutil.rmtree(tmp, ignore_errors=True)
    return True


if drive_ready():
    have = [p.name for p in BACKUP_DIR.glob("*") if (p / ".complete").exists()] if BACKUP_DIR.exists() else []
    print("💾 Drive 備份位置：", BACKUP_DIR, "｜已有：", have or "（尚無）")

# ---------- 1. ACE-Step 程式碼 ----------
if not (ACE_REPO / "pyproject.toml").exists():
    shutil.rmtree(ACE_REPO, ignore_errors=True)
    r = run_cmd(["git", "clone", "--depth", "1", ACE_REPO_URL, ACE_REPO], check=False)
    if r.returncode != 0 or not (ACE_REPO / "pyproject.toml").exists():
        print("⚠️ GitHub 無法使用，改用 Drive 備份")
        shutil.rmtree(ACE_REPO, ignore_errors=True)
        if not restore_code_tar("acestep_code", ACE_REPO):
            raise RuntimeError("無法取得 ACE-Step 程式碼（GitHub 與 Drive 備份都不可用）。")
        ACE_RESTORED.add("acestep_code")
elif FORCE_UPDATE_ACESTEP and (ACE_REPO / ".git").exists():
    run_cmd(["git", "-C", ACE_REPO, "pull", "--ff-only"], check=False)

# ---------- 2. 獨立的 Python 3.11 環境（uv） ----------
synced_now = False


def uv_sync():
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "uv"])
    # 官方的 Linux 加速套件（triton／flash-attn）只提供 Python 3.11 版
    run_cmd([sys.executable, "-m", "uv", "python", "install", "3.11"])
    run_cmd([sys.executable, "-m", "uv", "sync", "--python", "3.11"], cwd=str(ACE_REPO))
    SYNC_MARK.write_text(lock_hash())


if (FORCE_UPDATE_ACESTEP or not VENV_PY.exists() or not SYNC_MARK.exists()
        or SYNC_MARK.read_text().strip() != lock_hash()):
    print("\n📦 安裝 ACE-Step 套件（第一次約數分鐘）……")
    uv_sync()
    synced_now = True
else:
    print("✅ ACE-Step 套件已安裝")

# ---------- 3. 其他工具 ----------
if shutil.which("ffmpeg") is None:
    run_cmd(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# ---------- 4. 驗證 ----------
fresh_ok, fresh_log = fresh_ace_import_test()
if not fresh_ok and not synced_now:
    print("\n⚠️ import 測試失敗，重新安裝套件：\n" + fresh_log[-1500:])
    uv_sync()
    fresh_ok, fresh_log = fresh_ace_import_test()

print("\n===== import 測試 =====\n" + fresh_log[-1500:])
if not fresh_ok:
    raise RuntimeError("ACE-Step import 失敗，請保留上方完整輸出以便排查。")

print("\n✅ 環境就緒，請繼續執行下一格。")

In [ ]:
# @title ② 📁 掛載 Google Drive + GPU 檢查
from google.colab import drive

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("沒有偵測到 GPU，請到「執行階段 → 變更執行階段類型」選擇 A100。")
gpu_name, gpu_mem = [x.strip() for x in gpu.stdout.strip().splitlines()[0].split(",")]
gpu_mem_gb = int(gpu_mem) / 1024
print(f"GPU：{gpu_name}（{gpu_mem_gb:.1f} GB）")
if gpu_mem_gb < 23:
    print("⚠️ ACE-Step XL + 4B LM 官方建議至少 24GB VRAM，目前顯存可能不足。")

In [ ]:
# @title ③ 🎛️ 歌曲設定
import re

# @markdown ### 💾 儲存位置
OUTPUT_ROOT = "/content/drive/MyDrive/AI音樂/ACE-Step"  # @param {type:"string"}
PROJECT_NAME = "我的歌曲"  # @param {type:"string"}
# @markdown 檔名前綴：`song_v` → song_v01、song_v02……
FILE_PREFIX = "song_v"  # @param {type:"string"}
VERSION_DIGITS = 2  # @param {type:"integer"}

# @markdown ---
# @markdown ### 🎲 批量與 Seed
BATCH_COUNT = 3  # @param {type:"slider", min:1, max:20, step:1}
# @markdown `random` 隨機｜`fixed` 固定｜`sequence` 從固定值逐首 +1｜`list` 依清單
SEED_MODE = "random"  # @param ["random", "fixed", "sequence", "list"]
FIXED_SEED = 123456  # @param {type:"integer"}
SEED_LIST_TEXT = ""  # @param {type:"string"}

# @markdown ---
# @markdown ### 🎼 ACE-Step
# @markdown `xl-sft` 品質最高（預設 50 步）｜`xl-turbo` 快很多（預設 8 步）。LM 固定使用最強的 4B。改這裡要重新執行 ④。
DIT_MODEL = "acestep-v15-xl-sft"  # @param ["acestep-v15-xl-sft", "acestep-v15-xl-turbo"]
LM_MODEL = "acestep-5Hz-lm-4B"
# @markdown 歌詞語言；`unknown` 交給 LM 判斷
VOCAL_LANGUAGE = "zh"  # @param ["zh", "yue", "ja", "en", "ko", "unknown"]
# @markdown 長度（秒）：`-1` = 由 LM 依歌詞決定；或填 10～600
DURATION_SEC = -1  # @param {type:"integer"}
# @markdown 純音樂（忽略歌詞）
INSTRUMENTAL = False  # @param {type:"boolean"}
# @markdown LM 先規劃曲式與音樂結構（官方預設開啟；關掉比較快但結構比較鬆）
THINKING = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 📝 LRC
MAKE_LRC = True  # @param {type:"boolean"}
# @markdown `acestep` 內建對歌詞（快，不用安裝）｜`whisper` Demucs + Whisper（和 YuE2 版相同）
LRC_METHOD = "acestep"  # @param ["acestep", "whisper"]
# @markdown 以下只有 `whisper` 會用到
LRC_LANGUAGE = "zh"  # @param ["zh", "en", "ja"]
# @markdown 慢歌、長音多的歌建議勾選；Rap 等密集唱法可取消
LRC_NO_VAD = True  # @param {type:"boolean"}
# @markdown 對不上的行用推測時間補齊
LRC_INTERPOLATE = False  # @param {type:"boolean"}
LRC_WHISPER_MODEL = "medium"  # @param ["small", "medium", "large-v3"] {allow-input: true}
# @markdown `cpu` 最穩定；`cuda` 較快，失敗會自動改用 CPU
LRC_DEVICE = "cpu"  # @param ["cpu", "cuda"]
# @markdown 先用 Demucs 分離人聲再對歌詞，準確很多（第一次會自動安裝，約多花 1 分鐘）；取消則直接用完整混音
LRC_SEPARATE_VOCALS = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ### 🎧 輸出（FLAC 母檔一定保留）
MAKE_MP3 = True  # @param {type:"boolean"}
MP3_BITRATE = "320k"  # @param ["192k", "256k", "320k"]


SEED_LIST = [int(s) for s in re.split(r"[,\s，、]+", SEED_LIST_TEXT.strip()) if s]
if not (DURATION_SEC == -1 or 10 <= DURATION_SEC <= 600):
    raise ValueError("DURATION_SEC 需為 -1（自動）或 10～600 秒。")
if SEED_MODE == "list" and len(SEED_LIST) < BATCH_COUNT:
    raise ValueError(f"SEED_LIST 只有 {len(SEED_LIST)} 個，不夠生成 {BATCH_COUNT} 首。")

print("✅ 設定已讀取")

In [ ]:
# @title ③-2 ✍️ 曲風與歌詞

# CAPTION：曲風、樂器、人聲、情緒、速度感等（英文效果最穩，可以寫成一段描述，最多 512 字）
CAPTION = """
Mandopop ballad,
soft and clear female lead vocal,
gentle piano,
warm strings,
light drums,
emotional and nostalgic,
rainy night mood
""".strip()

# LYRICS：段落標籤可以保留；留空 = 純音樂（最多 4096 字）
LYRICS = """
[Verse 1]
窗外的雨 下了一整夜
咖啡涼了 我還在等你的回覆
螢幕亮了又暗
像我說不出口的心事

[Chorus]
如果明天天晴
我想陪你走過那條街
把沒說完的話
一句一句 都交給風

[Verse 2]
路燈把影子拉得好長
我數著腳步 假裝不想念
你說過的玩笑
還在耳邊 輕輕回響

[Chorus]
如果明天天晴
我想陪你走過那條街
把沒說完的話
一句一句 都交給風
""".strip()

if len(CAPTION) > 512:
    raise ValueError(f"CAPTION 最多 512 字，目前 {len(CAPTION)} 字。")
if len(LYRICS) > 4096:
    raise ValueError(f"LYRICS 最多 4096 字，目前 {len(LYRICS)} 字。")

print(f"專案：{PROJECT_NAME}｜批量：{BATCH_COUNT}｜Seed：{SEED_MODE}｜語言：{VOCAL_LANGUAGE}｜LRC：{LRC_METHOD if MAKE_LRC else '關'}")
print(f"CAPTION {len(CAPTION)} 字｜LYRICS {len(LYRICS.splitlines())} 行" + ("｜純音樂" if INSTRUMENTAL or not LYRICS else ""))

In [ ]:
# @title ③-3 🎚️ 進階設定（不確定就保持預設）
# @markdown 預設值與官方 Gradio 介面一致。
# @markdown ### 🎧 DiT 音訊生成
# @markdown 步數：`0` = 依模型預設（xl-sft 50｜xl-turbo 8）；xl-sft 可填 1～200，xl-turbo 1～20
INFERENCE_STEPS = 0  # @param {type:"integer"}
# @markdown CFG：只對 xl-sft 有效，turbo 會自動忽略（官方預設 7.0）
GUIDANCE_SCALE = 7.0  # @param {type:"number"}
# @markdown Timestep Shift：1～5（官方預設 3.0）
SHIFT = 3.0  # @param {type:"number"}
# @markdown `ode` 較快、同 seed 結果固定｜`sde` 隨機性較高
INFER_METHOD = "ode"  # @param ["ode", "sde"]

# @markdown ---
# @markdown ### 🧠 LM 規劃（Thinking 開啟時）
LM_TEMPERATURE = 0.85  # @param {type:"number"}
LM_CFG_SCALE = 2.0  # @param {type:"number"}
# @markdown Top K：`0` = 不限制
LM_TOP_K = 0  # @param {type:"integer"}
# @markdown Top P：`1.0` = 不限制
LM_TOP_P = 0.9  # @param {type:"number"}
LM_NEGATIVE_PROMPT = "NO USER INPUT"  # @param {type:"string"}
# @markdown 讓 LM 改寫、補充曲風描述（官方預設開啟；想完全照自己的 CAPTION 就取消）
USE_COT_CAPTION = True  # @param {type:"boolean"}


def resolved_steps(dit_model=None, steps=None):
    dit_model = dit_model or DIT_MODEL
    steps = INFERENCE_STEPS if steps is None else steps
    return int(steps) if int(steps) > 0 else (8 if "turbo" in dit_model else 50)


def validate_advanced():
    steps = resolved_steps()
    limit = 20 if "turbo" in DIT_MODEL else 200
    if not 1 <= steps <= limit:
        raise ValueError(f"{DIT_MODEL} 的步數需為 1～{limit}（0 = 預設）。")
    if not 1.0 <= float(GUIDANCE_SCALE) <= 15.0:
        raise ValueError("GUIDANCE_SCALE 需為 1～15（官方預設 7.0）。")
    if not 1.0 <= float(SHIFT) <= 5.0:
        raise ValueError("SHIFT 需為 1～5（官方預設 3.0）。")
    if not 0.0 <= float(LM_TEMPERATURE) <= 2.0:
        raise ValueError("LM_TEMPERATURE 需為 0～2。")
    if not 1.0 <= float(LM_CFG_SCALE) <= 3.0:
        raise ValueError("LM_CFG_SCALE 需為 1～3（1 = 不使用 CFG）。")
    if not 0 <= int(LM_TOP_K) <= 100:
        raise ValueError("LM_TOP_K 需為 0～100（0 = 不限制）。")
    if not 0.0 < float(LM_TOP_P) <= 1.0:
        raise ValueError("LM_TOP_P 需大於 0、最多 1.0。")


validate_advanced()
print(f"✅ 進階設定已讀取｜{DIT_MODEL}：{resolved_steps()} 步")

In [ ]:
# @title ③-4 🎼 曲式設定（可略過）
# @markdown 留空／0 = 交給 LM 決定（預設）；填了就照填的生成。圖形介面可以先讓 LM 規劃，再修改。
# @markdown BPM：`0` = 自動；可填 30～300
BPM = 0  # @param {type:"integer"}
# @markdown 調性，例如 `C major`、`A minor`、`F# minor`；留空 = 自動
KEYSCALE = ""  # @param {type:"string"}
# @markdown 拍號：`auto` 自動｜`4` = 4/4｜`3` = 3/4｜`2` = 2/4｜`6` = 6/8
TIMESIGNATURE = "auto"  # @param ["auto", "4", "3", "2", "6"]

if BPM and not 30 <= int(BPM) <= 300:
    raise ValueError("BPM 需為 0（自動）或 30～300。")
print("🎼 曲式：" + "｜".join([f"BPM {BPM or '自動'}", f"調性 {KEYSCALE.strip() or '自動'}",
                            f"拍號 {TIMESIGNATURE if TIMESIGNATURE != 'auto' else '自動'}"]))

In [ ]:
# @title 🧰 共用工具函式
import csv
import json
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import traceback
import uuid
from collections import deque
from datetime import datetime
from pathlib import Path

PROJECT_DIR = Path(OUTPUT_ROOT) / PROJECT_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ACE_TMP = Path("/content/acestep_tmp")
WORKER_PATH = Path("/content/acestep_worker.py")
MAIN_COMPONENTS = ["acestep-v15-turbo", "vae", "Qwen3-Embedding-0.6B", "acestep-5Hz-lm-1.7B"]


def next_version_number(root: Path, prefix: str) -> int:
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)$")
    found = [int(m.group(1)) for p in root.iterdir() if p.is_dir() and (m := pattern.match(p.name))]
    return max(found, default=0) + 1


def pick_seed(i: int) -> int:
    mode = SEED_MODE.lower().strip()
    if mode == "random":
        return random.SystemRandom().randint(1, 2_147_483_647)
    if mode == "fixed":
        return int(FIXED_SEED)
    if mode == "sequence":
        return int(FIXED_SEED) + i
    if mode == "list":
        return int(SEED_LIST[i])
    raise ValueError("SEED_MODE 必須是 random / fixed / sequence / list")


def normalize_timesig(value) -> str:
    """拍號統一成 ACE-Step 的寫法：4/4 → 4、6/8 → 6；auto／空白 → 空字串。"""
    text = str(value or "").strip().lower()
    if text in ("", "auto", "n/a", "0"):
        return ""
    m = re.match(r"^(\d+)", text)
    if not m or m.group(1) not in ("2", "3", "4", "6"):
        raise ValueError(f"拍號只能是 2、3、4、6（對應 2/4、3/4、4/4、6/8），收到：{value}")
    return m.group(1)


def timesig_label(value) -> str:
    return {"2": "2/4", "3": "3/4", "4": "4/4", "6": "6/8"}.get(str(value or ""), "自動")


def is_instrumental() -> bool:
    return bool(INSTRUMENTAL) or not str(LYRICS or "").strip()


def build_job(seed: int) -> dict:
    """把目前的設定整理成送給背景程序的工作。"""
    validate_advanced()
    if not str(CAPTION or "").strip():
        raise ValueError("請先填曲風描述（CAPTION）。")
    return {
        "cmd": "generate", "caption": CAPTION.strip(), "lyrics": LYRICS.strip(),
        "instrumental": is_instrumental(), "language": VOCAL_LANGUAGE,
        "duration": DURATION_SEC, "bpm": int(BPM or 0), "keyscale": str(KEYSCALE or "").strip(),
        "timesignature": normalize_timesig(TIMESIGNATURE),
        "steps": resolved_steps(), "guidance": float(GUIDANCE_SCALE), "shift": float(SHIFT),
        "infer_method": INFER_METHOD, "thinking": bool(THINKING),
        "lm_temperature": float(LM_TEMPERATURE), "lm_cfg_scale": float(LM_CFG_SCALE),
        "lm_top_k": int(LM_TOP_K), "lm_top_p": float(LM_TOP_P),
        "lm_negative_prompt": LM_NEGATIVE_PROMPT, "use_cot_caption": bool(USE_COT_CAPTION),
        "lrc_native": bool(MAKE_LRC) and LRC_METHOD == "acestep", "seed": int(seed),
    }


# ── 背景程序（在 ACE-Step 的 venv 裡跑，模型只載入一次） ──
WORKER_PATH.write_text(r'''
"""ACE-Step 背景程序：在 ACE-Step 自己的 Python 3.11 venv 裡執行。
模型只載入一次，之後從 stdin 一行一個 JSON 接工作，結果用標記行回傳。"""
import json
import os
import queue
import sys
import threading
import time
import traceback

# Colab 的 MPLBACKEND 指向 notebook 專用後端，venv 裡沒有，必須在 import 前改掉
os.environ["MPLBACKEND"] = "Agg"

MARK = "@@ACESTEP@@"
ROOT, DIT_MODEL, LM_MODEL = sys.argv[1], sys.argv[2], sys.argv[3]
sys.path.insert(0, ROOT)
os.chdir(ROOT)

_print_lock = threading.Lock()


def emit(kind, data=None):
    line = f"{MARK}{kind} " + json.dumps(data or {}, ensure_ascii=False, default=str)
    with _print_lock:
        print(line, flush=True)


def log(msg):
    with _print_lock:
        print(msg, flush=True)


try:
    from acestep.handler import AceStepHandler
    from acestep.llm_inference import LLMHandler
    from acestep.inference import GenerationParams, GenerationConfig, generate_music, format_sample
    from acestep.model_downloader import ensure_lm_model

    t0 = time.time()
    ckpt = os.path.join(ROOT, "checkpoints")

    log(f"載入 DiT：{DIT_MODEL}")
    dit = AceStepHandler()
    status, ok = dit.initialize_service(project_root=ROOT, config_path=DIT_MODEL, device="cuda")
    log(status)
    if not ok:
        raise RuntimeError(f"DiT 初始化失敗：{status}")

    log(f"載入 LM：{LM_MODEL}")
    ok, msg = ensure_lm_model(model_name=LM_MODEL, checkpoints_dir=ckpt)
    if not ok:
        raise RuntimeError(f"LM 下載失敗：{msg}")
    llm = LLMHandler()
    backend = "vllm"
    status, ok = llm.initialize(checkpoint_dir=ckpt, lm_model_path=LM_MODEL, backend=backend, device="cuda")
    if not ok:
        log(f"⚠️ vllm 後端失敗，改用 pt：{status}")
        backend = "pt"
        llm = LLMHandler()
        status, ok = llm.initialize(checkpoint_dir=ckpt, lm_model_path=LM_MODEL, backend=backend, device="cuda")
    if not ok:
        raise RuntimeError(f"LM 初始化失敗：{status}")
except Exception as e:
    traceback.print_exc()
    emit("FATAL", {"error": repr(e)})
    sys.exit(1)

emit("READY", {"dit": DIT_MODEL, "lm": LM_MODEL, "backend": backend,
               "load_seconds": round(time.time() - t0, 1)})

# ── stdin 讀取放在另一條執行緒，生成中也能收到「停止」 ──
jobs = queue.Queue()
cancel = threading.Event()
MAIN_THREAD = threading.current_thread()


def _reader():
    for raw in sys.stdin:
        raw = raw.strip()
        if not raw:
            continue
        try:
            msg = json.loads(raw)
        except json.JSONDecodeError:
            log(f"⚠️ 看不懂的指令：{raw[:200]}")
            continue
        if msg.get("cmd") == "cancel":
            cancel.set()
        else:
            jobs.put(msg)
    jobs.put({"cmd": "quit"})


threading.Thread(target=_reader, daemon=True).start()


class Cancelled(Exception):
    pass


def make_progress(job_id):
    state = {"last": 0.0}

    def progress(value=None, desc=None, *args, **kwargs):
        # 只有主執行緒的檢查點能中斷；ACE-Step 背景估算執行緒的例外會被它自己吞掉
        if cancel.is_set() and threading.current_thread() is MAIN_THREAD:
            raise Cancelled("使用者停止")
        now = time.time()
        if now - state["last"] < 0.5:
            return
        state["last"] = now
        try:
            v = float(value) if value is not None else None
        except (TypeError, ValueError):
            v = None
        emit("PROGRESS", {"id": job_id, "value": v, "desc": str(desc or "")})

    return progress


def none_if_zero(v):
    return None if v in (None, "", 0, 0.0) else v


def user_metadata(job):
    meta = {}
    if job.get("bpm"):
        meta["bpm"] = int(job["bpm"])
    if job.get("keyscale"):
        meta["keyscale"] = str(job["keyscale"])
    if job.get("timesignature"):
        meta["timesignature"] = str(job["timesignature"])
    if job.get("duration") and float(job["duration"]) > 0:
        meta["duration"] = int(float(job["duration"]))
    if job.get("language") and job["language"] != "unknown":
        meta["language"] = job["language"]
    return meta or None


def do_plan(job):
    top_p = job.get("lm_top_p")
    res = format_sample(
        llm, caption=job["caption"], lyrics=job["lyrics"] or "[Instrumental]",
        user_metadata=user_metadata(job),
        temperature=float(job.get("lm_temperature", 0.85)),
        top_k=none_if_zero(int(job.get("lm_top_k") or 0)),
        top_p=None if top_p is None or float(top_p) >= 1.0 else float(top_p),
    )
    if not res.success:
        emit("ERROR", {"id": job["id"], "error": res.error or res.status_message})
        return
    emit("PLAN", {"id": job["id"], "caption": res.caption, "bpm": res.bpm, "duration": res.duration,
                  "keyscale": res.keyscale, "timesignature": res.timesignature, "language": res.language})


def native_lrc(extra, duration_sec, language, steps):
    need = ["pred_latents", "encoder_hidden_states", "encoder_attention_mask", "context_latents", "lyric_token_idss"]
    if any(extra.get(k) is None for k in need):
        return None, "ACE-Step 沒有保留對歌詞需要的資料（可能是省記憶體模式）"
    r = dit.get_lyric_timestamp(
        pred_latent=extra["pred_latents"][0:1],
        encoder_hidden_states=extra["encoder_hidden_states"][0:1],
        encoder_attention_mask=extra["encoder_attention_mask"][0:1],
        context_latents=extra["context_latents"][0:1],
        lyric_token_ids=extra["lyric_token_idss"][0:1],
        total_duration_seconds=float(duration_sec),
        vocal_language=language or "en",
        inference_steps=int(steps),
        seed=42,
    )
    if r.get("success") and str(r.get("lrc_text") or "").strip():
        return r["lrc_text"], None
    return None, str(r.get("error") or "沒有產生任何時間軸")


def do_generate(job):
    jid = job["id"]
    instrumental = bool(job.get("instrumental"))
    lyrics = "[Instrumental]" if instrumental or not str(job.get("lyrics") or "").strip() else job["lyrics"]
    params = GenerationParams(
        task_type="text2music",
        caption=job["caption"],
        lyrics=lyrics,
        instrumental=instrumental,
        vocal_language=job.get("language") or "unknown",
        bpm=job.get("bpm") or None,
        keyscale=job.get("keyscale") or "",
        timesignature=job.get("timesignature") or "",
        duration=float(job.get("duration") or -1),
        inference_steps=int(job["steps"]),
        guidance_scale=float(job["guidance"]),
        shift=float(job["shift"]),
        infer_method=job.get("infer_method", "ode"),
        thinking=bool(job.get("thinking", True)),
        lm_temperature=float(job.get("lm_temperature", 0.85)),
        lm_cfg_scale=float(job.get("lm_cfg_scale", 2.0)),
        lm_top_k=int(job.get("lm_top_k") or 0),
        lm_top_p=float(job.get("lm_top_p", 0.9)),
        lm_negative_prompt=job.get("lm_negative_prompt") or "NO USER INPUT",
        use_cot_caption=bool(job.get("use_cot_caption", True)),
    )
    gconf = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[int(job["seed"])], audio_format="flac")
    os.makedirs(job["out_dir"], exist_ok=True)
    t1 = time.time()
    res = generate_music(dit, llm, params, gconf, save_dir=job["out_dir"], progress=make_progress(jid))
    if cancel.is_set():
        emit("ERROR", {"id": jid, "error": "使用者停止", "cancelled": True})
        return
    if not res.success or not res.audios:
        emit("ERROR", {"id": jid, "error": str(res.error or res.status_message)})
        return
    audio = res.audios[0]
    extra = res.extra_outputs or {}
    tensor = audio.get("tensor")
    sr = audio.get("sample_rate") or 48000
    duration_sec = (tensor.shape[-1] / sr) if tensor is not None else None
    meta = extra.get("lm_metadata") if isinstance(extra.get("lm_metadata"), dict) else {}
    out = {
        "id": jid,
        "path": audio.get("path", ""),
        "seed": (audio.get("params") or {}).get("seed", job["seed"]),
        "duration_sec": duration_sec,
        "lm_metadata": meta,
        "time_costs": extra.get("time_costs"),
        "seconds": round(time.time() - t1, 1),
    }
    if job.get("lrc_native") and not instrumental:
        language = job.get("language")
        if not language or language == "unknown":
            language = meta.get("language") or meta.get("vocal_language") or "en"
        try:
            emit("PROGRESS", {"id": jid, "value": 1.0, "desc": "LRC"})
            lrc, err = native_lrc(extra, duration_sec or float(job.get("duration") or 0) or 60.0,
                                  language, int(job["steps"]))
            out["lrc_text"], out["lrc_error"] = lrc, err
        except Exception as e:
            traceback.print_exc()
            out["lrc_text"], out["lrc_error"] = None, repr(e)
    emit("DONE", out)


while True:
    job = jobs.get()
    cmd = job.get("cmd")
    if cmd == "quit":
        break
    cancel.clear()
    try:
        if cmd == "plan":
            do_plan(job)
        elif cmd == "generate":
            do_generate(job)
        else:
            emit("ERROR", {"id": job.get("id"), "error": f"未知指令：{cmd}"})
    except Cancelled:
        emit("ERROR", {"id": job.get("id"), "error": "使用者停止", "cancelled": True})
    except Exception as e:
        traceback.print_exc()
        emit("ERROR", {"id": job.get("id"), "error": repr(e), "cancelled": cancel.is_set()})
'''.lstrip(), encoding="utf-8")

PROGRESS_LABELS = [
    ("phase 1", "LM 規劃曲式"), ("phase 2", "LM 生成音樂結構"), ("preparing inputs", "準備輸入"),
    ("generating music", "生成音訊"), ("decoding", "解碼音訊"), ("preparing audio", "整理音訊"),
    ("lrc", "對歌詞（LRC）"),
]


def progress_label(desc: str) -> str:
    low = str(desc or "").lower()
    for key, label in PROGRESS_LABELS:
        if key in low:
            return label
    return str(desc or "生成中")


class AceWorker:
    MARK = "@@ACESTEP@@"

    def __init__(self):
        self.proc = None
        self.sig = None
        self.info = {}
        self.logs = deque(maxlen=300)
        self.io_lock = threading.Lock()
        self.write_lock = threading.Lock()

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def _write(self, msg):
        with self.write_lock:
            self.proc.stdin.write(json.dumps(msg, ensure_ascii=False) + "\n")
            self.proc.stdin.flush()

    def _read_until(self, want, job_id=None, on_progress=None, echo=True):
        while True:
            line = self.proc.stdout.readline()
            if not line:
                raise RuntimeError(f"背景程序已結束（exit {self.proc.poll()}）。最後的訊息：\n" + self.recent_logs(15))
            if line.startswith(self.MARK):
                kind, _, payload = line[len(self.MARK):].partition(" ")
                kind, data = kind.strip(), json.loads(payload)
                if job_id is not None and data.get("id") not in (None, job_id):
                    continue          # 之前中斷的工作留下的訊息，略過
                if kind == "PROGRESS":
                    if on_progress:
                        on_progress(data.get("value"), data.get("desc", ""))
                    continue
                if kind in want:
                    return kind, data
                continue
            text = line.rstrip("\n").split("\r")[-1]
            if text.strip():
                self.logs.append(text)
                if echo:
                    print(text)

    def recent_logs(self, n=40):
        return "\n".join(list(self.logs)[-n:])

    def start(self, dit_model, lm_model):
        self.stop()
        self.logs.clear()
        self.proc = subprocess.Popen(
            [str(VENV_PY), "-u", str(WORKER_PATH), str(ACE_REPO), dit_model, lm_model],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, cwd=str(ACE_REPO), env=venv_env())
        kind, info = self._read_until({"READY", "FATAL"})
        if kind != "READY":
            self.proc.wait()
            raise RuntimeError(f"模型載入失敗：{info.get('error')}\n" + self.recent_logs(20))
        self.sig, self.info = (dit_model, lm_model), info
        return info

    def request(self, job, on_progress=None, echo=True):
        if not self.alive():
            raise RuntimeError("背景模型沒有在跑，請先執行 ④。")
        job = dict(job, id=uuid.uuid4().hex)
        with self.io_lock:
            self._write(job)
            try:
                return self._read_until({"DONE", "ERROR", "PLAN"}, job["id"], on_progress, echo)
            except KeyboardInterrupt:
                self.cancel()
                raise

    def cancel(self):
        if self.alive():
            try:
                self._write({"cmd": "cancel"})
            except Exception:
                pass

    def stop(self):
        if self.alive():
            try:
                self._write({"cmd": "quit"})
                self.proc.wait(timeout=30)
            except Exception:
                self.proc.kill()
        self.proc, self.sig = None, None


# 重新執行這格時，沿用已經在背景跑的模型，不必重新載入
_old_ace = globals().get("ACE")
ACE = AceWorker()
if _old_ace is not None and getattr(_old_ace, "proc", None) is not None and _old_ace.proc.poll() is None:
    ACE.proc, ACE.sig, ACE.info = _old_ace.proc, _old_ace.sig, getattr(_old_ace, "info", {})


# ── 音訊與 LRC ──
def audio_duration(path: Path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(path)],
                       text=True, capture_output=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return None


def make_mp3(flac_path: Path, mp3_path: Path):
    r = subprocess.run(["ffmpeg", "-y", "-hide_banner", "-loglevel", "error", "-i", str(flac_path),
                        "-codec:a", "libmp3lame", "-b:a", MP3_BITRATE, str(mp3_path)],
                       text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return r.returncode == 0, r.stdout


def count_lrc_lines(text: str) -> int:
    return sum(1 for line in str(text or "").splitlines() if re.match(r"^\[\d+:\d+(?:\.\d+)?\]\s*\S", line.strip()))


def lyric_line_count(lyrics: str) -> int:
    return sum(1 for line in str(lyrics or "").splitlines() if line.strip() and not re.match(r"^\[.*\]$", line.strip()))


DEMUCS_ENV = Path("/content/demucs_env")
STEM_ROOT = Path("/content/acestep_vocal_stems")
LAST_LRC_INFO = {}      # 最近一次 LRC 的結果：對上幾行、有沒有用人聲檔


def ensure_lyric_align():
    """whisper LRC 第一次用到才安裝 lyric-align（裝在 Colab 內建的 Python，不影響 ACE-Step）。"""
    if shutil.which("lyric-align"):
        return
    print("⏳ 第一次使用 whisper LRC：安裝 lyric-align（約 1 分鐘）……")
    for args in (["lyric-align[asr]"], ["av<19"]):      # PyAV 19 與 faster-whisper 不相容
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args],
                           text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            raise RuntimeError(f"lyric-align 安裝失敗：{(r.stdout or '')[-400:]}")


def ensure_demucs():
    """Demucs 裝在獨立環境，第一次用到才安裝。"""
    py = DEMUCS_ENV / "bin" / "python"
    if py.exists() and (DEMUCS_ENV / ".ready").exists():
        return py
    print("⏳ 第一次分離人聲：安裝 Demucs（獨立環境，約 1 分鐘）……")
    steps = [[sys.executable, "-m", "pip", "install", "-q", "uv"],
             [sys.executable, "-m", "uv", "venv", str(DEMUCS_ENV), "--python", "3.11", "-q"],
             [sys.executable, "-m", "uv", "pip", "install", "-q", "--python", str(py),
              "torch==2.5.1", "torchaudio==2.5.1", "demucs==4.0.1", "soundfile"]]
    for cmd in steps:
        r = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        if r.returncode != 0:
            raise RuntimeError(f"Demucs 安裝失敗：{(r.stdout or '')[-400:]}")
    (DEMUCS_ENV / ".ready").write_text("ok")
    return py


def separate_vocals(audio_path):
    """用 Demucs 分離出人聲，回傳人聲檔路徑。先試 GPU，失敗（例如顯存被 ACE-Step 占滿）再用 CPU。"""
    import hashlib
    audio_path = Path(audio_path)
    out = STEM_ROOT / hashlib.md5(str(audio_path).encode("utf-8")).hexdigest()[:12]
    stem = out / "htdemucs" / "audio" / "vocals.wav"
    if stem.exists():
        return stem
    py = ensure_demucs()
    out.mkdir(parents=True, exist_ok=True)
    local = out / f"audio{audio_path.suffix}"      # 複製到本機：路徑不含中文，也比直接讀雲端硬碟快
    shutil.copy2(audio_path, local)
    log = ""
    for device in (("cuda", "cpu") if shutil.which("nvidia-smi") else ("cpu",)):
        r = subprocess.run([str(py), "-m", "demucs", "-n", "htdemucs", "--two-stems", "vocals", "-d", device,
                            "-o", str(out), str(local)], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        log = r.stdout or ""
        if r.returncode == 0 and stem.exists():
            return stem
    raise RuntimeError(f"Demucs 沒有產生人聲檔：{log[-300:]}")


def run_lrc(audio_path, lyrics_path, lrc_path, segments_path):
    """whisper LRC。回傳 (是否至少對上一行, 紀錄文字)；實際對上幾行記在 LAST_LRC_INFO。"""
    global LAST_LRC_INFO
    notes, align_audio = [], audio_path
    try:
        ensure_lyric_align()
    except Exception as e:
        LAST_LRC_INFO = {"matched": None, "total": None, "separated": False}
        return False, f"⚠️ {e}"
    # 直接拿完整混音去辨識，Whisper 常把整段合併成少數幾段，結果一行都對不上；先分離人聲準確很多。
    if globals().get("LRC_SEPARATE_VOCALS", True):
        try:
            align_audio = separate_vocals(audio_path)
            notes.append(f"vocal stem (Demucs): {align_audio}")
        except Exception as e:
            notes.append(f"⚠️ 人聲分離失敗，改用完整混音對歌詞：{e}")
            print(notes[-1])

    def build(device):
        cmd = ["lyric-align", str(align_audio), str(lyrics_path), "--language", LRC_LANGUAGE,
               "--model", LRC_WHISPER_MODEL, "--device", device,
               "--dump-segments", str(segments_path), "-o", str(lrc_path)]
        if LRC_NO_VAD:
            cmd.append("--no-vad")
        if LRC_INTERPOLATE:
            cmd.append("--interpolate")
        return cmd

    def execute(device):
        try:
            return subprocess.run(build(device), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        except FileNotFoundError as e:
            return subprocess.CompletedProcess(build(device), 127, stdout=f"找不到 lyric-align：{e}")

    r = execute(LRC_DEVICE)
    if r.returncode != 0 and LRC_DEVICE == "cuda":
        if any(m in r.stdout.lower() for m in ["cublas", "cudnn", "cuda error", "cuda runtime", "cannot be loaded", "out of memory"]):
            print("⚠️ GPU 版 Whisper 無法使用，改用 CPU")
            r = execute("cpu")
    found = re.search(r"aligned (\d+)/(\d+) lines", r.stdout or "")
    matched, total = (int(found.group(1)), int(found.group(2))) if found else (None, None)
    LAST_LRC_INFO = {"matched": matched, "total": total, "separated": align_audio is not audio_path}
    # 指令成功但一行都沒對上，不算成功。
    return r.returncode == 0 and matched != 0, "\n".join(notes + [r.stdout or ""])


def save_params_txt(path, song_name, seed, job, result, duration_sec, lrc_ok, mp3_ok):
    meta = result.get("lm_metadata") or {}
    lrc_desc = (f"{LRC_METHOD}" + (f"（{LRC_LANGUAGE}，{LRC_WHISPER_MODEL}，no-VAD {LRC_NO_VAD}，interpolate {LRC_INTERPOLATE}，"
                                   f"人聲分離 {LRC_SEPARATE_VOCALS}）" if LRC_METHOD == "whisper" else ""))
    shown = {k: v for k, v in job.items() if k not in ("cmd", "id", "caption", "lyrics")}
    path.write_text(f"""歌曲：{song_name}
生成時間：{datetime.now().isoformat(timespec="seconds")}

模型：{DIT_MODEL} + {LM_MODEL}｜步數：{job['steps']}｜CFG：{job['guidance']}｜Shift：{job['shift']}｜{job['infer_method']}
Seed：{seed}（{SEED_MODE}）｜語言：{job['language']}｜Thinking：{job['thinking']}｜純音樂：{job['instrumental']}
指定曲式：BPM {job['bpm'] or '自動'}｜調性 {job['keyscale'] or '自動'}｜拍號 {timesig_label(job['timesignature'])}｜長度 {job['duration'] if job['duration'] and job['duration'] > 0 else '自動'}
LM 決定：BPM {meta.get('bpm', '—')}｜調性 {meta.get('keyscale', '—')}｜拍號 {timesig_label(meta.get('timesignature')) if meta.get('timesignature') else '—'}｜長度 {meta.get('duration', '—')}
實際長度：{duration_sec:.2f} 秒｜生成耗時：{result.get('seconds')} 秒
LRC：{"成功" if lrc_ok else "失敗或未啟用"}（{lrc_desc}）
MP3：{"成功" if mp3_ok else "未輸出"}（{MP3_BITRATE}）

===== CAPTION =====
{job['caption']}

===== LM 改寫後的 CAPTION =====
{meta.get('caption') or '（沒有改寫）'}

===== LYRICS =====
{job['lyrics'] or '（純音樂）'}

===== 參數 =====
{json.dumps(shown, ensure_ascii=False, indent=2)}
""", encoding="utf-8")


def write_failed(project_dir, song_name, job, message):
    fail_dir = Path(project_dir) / f"{song_name}_FAILED"
    fail_dir.mkdir(parents=True, exist_ok=True)
    (fail_dir / "ERROR.txt").write_text(f"{message}\n\n===== 背景程序最後的訊息 =====\n{ACE.recent_logs(60)}",
                                        encoding="utf-8")
    (fail_dir / "request.json").write_text(json.dumps(job, ensure_ascii=False, indent=2), encoding="utf-8")
    return fail_dir


def generate_song(project_dir, song_name, seed, on_progress=None, echo=True):
    """生成一首歌並整理檔案。表單批量和圖形介面共用。
    回傳 dict：row（摘要列）、audio、lrc_text、notes；被停止時回傳 None。"""
    project_dir = Path(project_dir)
    output_dir = project_dir / song_name
    if output_dir.exists():
        raise RuntimeError(f"{output_dir} 已存在，停止以免覆蓋。")
    job = build_job(seed)
    job["out_dir"] = str(ACE_TMP / uuid.uuid4().hex)
    kind, res = ACE.request(job, on_progress=on_progress, echo=echo)
    if kind == "ERROR":
        shutil.rmtree(job["out_dir"], ignore_errors=True)
        if res.get("cancelled"):
            return None
        fail_dir = write_failed(project_dir, song_name, job, f"生成失敗：{res.get('error')}")
        return {"row": {"song": song_name, "seed": seed, "duration_sec": "", "lrc": "", "mp3_ok": False,
                        "folder": str(fail_dir), "status": f"失敗：{res.get('error')}"[:120]},
                "audio": None, "lrc_text": "", "notes": []}
    src = Path(res.get("path") or "")
    if not src.is_file():
        fail_dir = write_failed(project_dir, song_name, job, f"生成完成但找不到音檔：{src}")
        return {"row": {"song": song_name, "seed": seed, "duration_sec": "", "lrc": "", "mp3_ok": False,
                        "folder": str(fail_dir), "status": "失敗：找不到音檔"}, "audio": None, "lrc_text": "", "notes": []}

    output_dir.mkdir(parents=True)
    named_flac = output_dir / f"{song_name}.flac"
    shutil.move(str(src), named_flac)
    shutil.rmtree(src.parent, ignore_errors=True)
    lyrics_path = output_dir / f"{song_name}_lyrics.txt"
    lyrics_path.write_text(job["lyrics"], encoding="utf-8")
    duration = res.get("duration_sec") or audio_duration(named_flac) or 0.0
    seed = res.get("seed", seed)
    notes = []

    # LRC
    lrc_ok, lrc_label, lrc_text = False, "", ""
    lrc_path = output_dir / f"{song_name}.lrc"
    if MAKE_LRC and not job["instrumental"]:
        if LRC_METHOD == "acestep":
            if res.get("lrc_text"):
                lrc_text = res["lrc_text"].strip()
                lrc_path.write_text(lrc_text + "\n", encoding="utf-8")
                n = count_lrc_lines(lrc_text)
                lrc_ok, lrc_label = n > 0, f"{n} 行"     # 內建 LRC 會自己斷句，行數不一定和歌詞一樣
            else:
                (output_dir / "lrc_log.txt").write_text(str(res.get("lrc_error")), encoding="utf-8")
                notes.append(f"⚠️ {song_name} 內建 LRC 失敗：{res.get('lrc_error')}（可用 ⑤B 以 whisper 補做）")
        else:
            if on_progress:
                on_progress(1.0, "lrc")
            lrc_ok, lrc_log = run_lrc(named_flac, lyrics_path, lrc_path, output_dir / f"{song_name}_segments.json")
            (output_dir / "lrc_log.txt").write_text(lrc_log, encoding="utf-8")
            if echo:
                print(lrc_log)
            info = LAST_LRC_INFO or {}
            if info.get("total"):
                lrc_label = f"{info['matched']}/{info['total']} 行"
            lrc_text = lrc_path.read_text(encoding="utf-8-sig").strip() if lrc_ok and lrc_path.exists() else ""

    # MP3
    mp3_ok = False
    if MAKE_MP3:
        mp3_ok, mp3_log = make_mp3(named_flac, output_dir / f"{song_name}.mp3")
        if not mp3_ok:
            (output_dir / "mp3_log.txt").write_text(mp3_log, encoding="utf-8")
            notes.append(f"⚠️ {song_name} MP3 轉檔失敗")

    save_params_txt(output_dir / "參數.txt", song_name, seed, job, res, duration, lrc_ok, mp3_ok)
    meta = res.get("lm_metadata") or {}
    (output_dir / "params.json").write_text(json.dumps({
        "song": song_name, "seed": seed, "dit_model": DIT_MODEL, "lm_model": LM_MODEL, "request": job,
        "lm_metadata": meta, "duration_sec": duration, "generate_seconds": res.get("seconds"),
        "time_costs": res.get("time_costs"), "lrc_method": LRC_METHOD if MAKE_LRC else None,
        "created": datetime.now().isoformat(timespec="seconds")}, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8")
    mp3 = output_dir / f"{song_name}.mp3"
    return {"row": {"song": song_name, "seed": seed, "duration_sec": round(duration, 2),
                    "lrc": lrc_label or ("✓" if lrc_ok else ("✗" if MAKE_LRC and not job["instrumental"] else "—")),
                    "mp3_ok": mp3_ok, "folder": str(output_dir), "status": "ok"},
            "audio": str(mp3 if mp3.exists() else named_flac), "lrc_text": lrc_text, "notes": notes,
            "meta": meta}


def write_batch_csv(project_dir, rows):
    summary = Path(project_dir) / f"batch_{datetime.now():%Y%m%d_%H%M%S}.csv"
    with summary.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=["song", "seed", "duration_sec", "lrc", "mp3_ok", "folder", "status"])
        w.writeheader()
        w.writerows(rows)
    return summary


def plan_structure(on_progress=None, echo=True):
    """只讓 LM 規劃曲式（BPM／調性／拍號／長度），不生成歌曲。"""
    job = build_job(0)
    job["cmd"] = "plan"
    kind, res = ACE.request(job, on_progress=on_progress, echo=echo)
    if kind != "PLAN":
        if res.get("cancelled"):
            return None
        raise RuntimeError(f"LM 規劃失敗：{res.get('error')}")
    return res


print("✅ 工具就緒｜專案資料夾：", PROJECT_DIR)

In [ ]:
# @title ④ 🚀 載入 ACE-Step
# 同一個 Runtime 只需載入一次；模型會留在背景程序裡。第一次會下載模型，比較久。
CKPT_DIR = ACE_REPO / "checkpoints"
WEIGHT_FILES = ("model.safetensors", "model.safetensors.index.json", "pytorch_model.bin",
                "pytorch_model.bin.index.json", "diffusion_pytorch_model.safetensors",
                "diffusion_pytorch_model.safetensors.index.json")
DOWNLOAD_CODE = r'''
import os, sys
os.environ["MPLBACKEND"] = "Agg"
root, dit, lm = sys.argv[1:4]
sys.path.insert(0, root)
from pathlib import Path
from acestep.model_downloader import ensure_main_model, ensure_dit_model, ensure_lm_model
ck = Path(root) / "checkpoints"
ok = True
for label, fn in (("主模型", lambda: ensure_main_model(ck)), (dit, lambda: ensure_dit_model(dit, ck)),
                  (lm, lambda: ensure_lm_model(lm, ck))):
    try:
        s, m = fn()
    except Exception as e:
        s, m = False, repr(e)
    print(("✅ " if s else "❌ ") + label + "：" + str(m), flush=True)
    ok = ok and s
sys.exit(0 if ok else 1)
'''


def needed_models():
    return MAIN_COMPONENTS + [DIT_MODEL, LM_MODEL]


def missing_models():
    return [n for n in needed_models() if not any((CKPT_DIR / n / f).exists() for f in WEIGHT_FILES)]


if ACE.alive() and ACE.sig == (DIT_MODEL, LM_MODEL):
    print(f"✅ ACE-Step 已在背景載入（{DIT_MODEL} + {LM_MODEL}），略過")
else:
    restored_now = []
    missing = missing_models()
    if missing:
        print("⬇️ 下載模型：", "、".join(missing))
        subprocess.run([str(VENV_PY), "-c", DOWNLOAD_CODE, str(ACE_REPO), DIT_MODEL, LM_MODEL],
                       cwd=str(ACE_REPO), env=venv_env())
        missing = missing_models()
        if missing:
            print(f"⚠️ 從 Hugging Face 下載失敗：{missing}\n→ 改用 Drive 備份")
            for name in missing:
                if restore_dirs(f"ckpt_{name}", CKPT_DIR):
                    ACE_RESTORED.add(f"ckpt_{name}")
                    restored_now.append(name)
            missing = missing_models()
            if missing:
                raise RuntimeError(f"無法取得模型：{missing}（Hugging Face 與 Drive 備份都不可用）。")
    if ACE.alive():
        print(f"🔄 模型設定改變（{ACE.sig[0]} → {DIT_MODEL}），重新載入……")
    print(f"載入 {DIT_MODEL} + {LM_MODEL}（約數分鐘）……")
    info = ACE.start(DIT_MODEL, LM_MODEL)
    print(f"✅ 載入完成（{info['load_seconds']} 秒，LM 後端：{info['backend']}）"
          + (f"（{'、'.join(restored_now)} 來自 Drive 備份）" if restored_now else ""))

## ⑤G 圖形介面（選用）

想用圖形介面就保持 `USE_GUI` 勾選（必須先執行 ④）。勾選時，下面的 ⑤ 表單批量生成會自動略過；想用原本的表單流程就取消勾選。

- **曲風、歌詞都是文字框**，直接貼上就能用，不用改程式碼。面板開啟時會帶入 ③ 各格目前的設定值。
- **專案名稱和歌曲名稱都能在面板上改**：專案名稱是資料夾，歌曲名稱後面會自動加編號（例如 `song_v01`），可以用中文。
- **先看曲式再生成**：按「先讓 LM 規劃曲式」只會規劃 BPM、調性、拍號和長度（很快），結果會填進曲式欄，可以直接修改再按「生成歌曲」。曲式欄留空或「自動」的項目，每首都會交給 LM 決定。LM 改寫後的曲風描述會顯示在狀態欄，想用的話按「套用 LM 改寫的曲風描述」。
- **停止**：會在下一個檢查點停下，已完成的歌會保留。
- **歌曲瀏覽**：回聽之前的歌、看 LRC 和參數，也可以把某一首的曲式（或曲式＋Seed）帶回「生成」分頁，修改後重新生成。
- 勾選 `OPEN_IN_NEW_TAB`：這格會印出 `gradio.live` 網址和帳號密碼，在新分頁登入使用。網址是公開的，請不要把帳號密碼分享出去。取消勾選則內嵌在這格下方，不需要密碼。如果拿不到臨時網址，會自動改用內嵌模式。
- **免費版 Colab 請注意**：Colab 規定免費執行階段不能主要透過網頁介面操作，使用圖形介面可能被中斷。付費方案且運算單元有餘額時不受此限。
- 面板開著的期間請保持 Colab 分頁開啟；用完請到「執行階段 → 中斷連線並刪除執行階段」。

> 圖形介面跑在 Colab 內建的 Python 裡（Gradio 5），模型在背景程序裡，兩邊互不影響。

In [ ]:
# @title ⑤G 🖥️ 圖形介面（Gradio）
# @markdown 勾選後執行這格會開啟圖形介面，下面的 ⑤ 表單批量生成會自動略過。必須先執行 ④。
USE_GUI = True # @param {type:"boolean"}
# @markdown **另開分頁**：產生 `gradio.live` 臨時網址，用這格印出的帳號密碼登入。取消勾選則直接內嵌在這格下方（不需要密碼）。
OPEN_IN_NEW_TAB = True # @param {type:"boolean"}
# @markdown 密碼留空的話，每次執行會自動產生一組新的。
GUI_PASSWORD = "" # @param {type:"string"}

ACESTEP_GUI_ACTIVE = bool(USE_GUI)


def _acestep_gui_main():
    import importlib
    import importlib.metadata
    import json
    import random
    import re
    import secrets
    import subprocess
    import sys
    import threading
    import time
    import traceback
    from datetime import datetime
    from pathlib import Path

    G = globals()
    if G.get("ACE") is None or not G["ACE"].alive():
        raise RuntimeError("請先執行 ④ 載入 ACE-Step，再執行 ⑤G。")
    for needed in ("generate_song", "plan_structure", "build_job", "pick_seed", "next_version_number",
                   "write_batch_csv", "normalize_timesig", "validate_advanced"):
        if needed not in G:
            raise RuntimeError("找不到共用工具函式，請先執行「🧰 共用工具函式」那一格。")

    # ── Gradio 5（裝在 Colab 內建的 Python，不影響 ACE-Step 的 venv） ──
    def installed(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None

    gradio_ver = installed("gradio")
    if gradio_ver is None or gradio_ver.split(".")[0] != "5":
        print(f"安裝 Gradio 5（目前：{gradio_ver or '未安裝'}），約一分鐘…", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gradio==5.50.0"], check=True)
        importlib.invalidate_caches()
        for name in [m for m in sys.modules if m.split(".")[0] in ("gradio", "gradio_client")]:
            del sys.modules[name]
    import gradio as gr

    ACE = G["ACE"]
    output_root = Path(G.get("OUTPUT_ROOT", "/content/drive/MyDrive/AI音樂/ACE-Step"))
    output_root.mkdir(parents=True, exist_ok=True)
    cancel = threading.Event()
    TIMESIG_CHOICES = [("自動", "auto"), ("4/4", "4"), ("3/4", "3"), ("2/4", "2"), ("6/8", "6")]
    LANG_CHOICES = ["zh", "yue", "ja", "en", "ko", "unknown"]
    LRC_EMPTY = ("（LRC 是空的：沒有對上任何一行歌詞。可以把 LRC 方式改成 whisper 再用 ⑤B 補做；"
                 "歌曲太短或幾乎沒唱歌詞時也會這樣。詳見歌曲資料夾裡的 lrc_log.txt）")

    def as_int(value, name, default=None):
        if value is None or value == "":
            if default is None:
                raise gr.Error(f"{name} 不能空白。")
            return default
        try:
            return int(round(float(value)))
        except (TypeError, ValueError):
            raise gr.Error(f"{name} 必須是數字。") from None

    def as_float(value, name, default):
        if value is None or value == "":
            return default
        try:
            return float(value)
        except (TypeError, ValueError):
            raise gr.Error(f"{name} 必須是數字。") from None

    KEYS = ["caption", "lyrics", "instrumental", "language", "duration", "thinking",
            "batch", "seed_mode", "fixed_seed", "seed_list",
            "bpm", "keyscale", "timesig",
            "steps", "guidance", "shift", "infer_method",
            "lm_temp", "lm_cfg", "lm_top_k", "lm_top_p", "lm_neg", "cot_caption",
            "make_lrc", "lrc_method", "lrc_lang", "lrc_no_vad", "lrc_interp", "lrc_model", "lrc_device",
            "lrc_separate", "make_mp3", "mp3_bitrate"]

    def apply_settings(v):
        """把面板上的值寫回 notebook 的設定變數，之後沿用原本表單流程的函式。"""
        duration = as_int(v["duration"], "長度", -1)
        if not (duration == -1 or 10 <= duration <= 600):
            raise gr.Error("長度需為 -1（自動）或 10～600 秒。")
        bpm = as_int(v["bpm"], "BPM", 0)
        if bpm and not 30 <= bpm <= 300:
            raise gr.Error("BPM 需為 0（自動）或 30～300。")
        seed_list = [int(s) for s in re.split(r"[,\s，、]+", str(v["seed_list"] or "").strip()) if s.strip().lstrip("-").isdigit()]
        batch = as_int(v["batch"], "批量數", 1)
        if v["seed_mode"] == "list" and len(seed_list) < batch:
            raise gr.Error(f"Seed 清單只有 {len(seed_list)} 個，不夠生成 {batch} 首。")
        caption = str(v["caption"] or "").strip()
        lyrics = str(v["lyrics"] or "").strip()
        if not caption:
            raise gr.Error("請先填曲風描述（CAPTION）。")
        if len(caption) > 512:
            raise gr.Error(f"曲風描述最多 512 字，目前 {len(caption)} 字。")
        if len(lyrics) > 4096:
            raise gr.Error(f"歌詞最多 4096 字，目前 {len(lyrics)} 字。")
        try:
            timesig = G["normalize_timesig"](v["timesig"])
        except ValueError as e:
            raise gr.Error(str(e)) from None
        G.update(
            CAPTION=caption, LYRICS=lyrics, INSTRUMENTAL=bool(v["instrumental"]),
            VOCAL_LANGUAGE=v["language"] or "unknown", DURATION_SEC=duration, THINKING=bool(v["thinking"]),
            BATCH_COUNT=batch, SEED_MODE=v["seed_mode"], FIXED_SEED=as_int(v["fixed_seed"], "固定 Seed", 123456),
            SEED_LIST=seed_list, SEED_LIST_TEXT=str(v["seed_list"] or ""),
            BPM=bpm, KEYSCALE=str(v["keyscale"] or "").strip(), TIMESIGNATURE=timesig or "auto",
            INFERENCE_STEPS=as_int(v["steps"], "步數", 0), GUIDANCE_SCALE=as_float(v["guidance"], "CFG", 7.0),
            SHIFT=as_float(v["shift"], "Shift", 3.0), INFER_METHOD=v["infer_method"] or "ode",
            LM_TEMPERATURE=as_float(v["lm_temp"], "LM Temperature", 0.85),
            LM_CFG_SCALE=as_float(v["lm_cfg"], "LM CFG", 2.0), LM_TOP_K=as_int(v["lm_top_k"], "LM Top K", 0),
            LM_TOP_P=as_float(v["lm_top_p"], "LM Top P", 0.9),
            LM_NEGATIVE_PROMPT=str(v["lm_neg"] or "NO USER INPUT"), USE_COT_CAPTION=bool(v["cot_caption"]),
            MAKE_LRC=bool(v["make_lrc"]), LRC_METHOD=v["lrc_method"] or "acestep", LRC_LANGUAGE=v["lrc_lang"],
            LRC_NO_VAD=bool(v["lrc_no_vad"]), LRC_INTERPOLATE=bool(v["lrc_interp"]),
            LRC_WHISPER_MODEL=str(v["lrc_model"]).strip() or "medium", LRC_DEVICE=v["lrc_device"],
            LRC_SEPARATE_VOCALS=bool(v["lrc_separate"]), MAKE_MP3=bool(v["make_mp3"]), MP3_BITRATE=v["mp3_bitrate"],
        )
        try:
            G["validate_advanced"]()
        except (ValueError, TypeError) as e:
            raise gr.Error(str(e)) from None
        if ACE.sig != (G["DIT_MODEL"], G["LM_MODEL"]):
            raise gr.Error("③ 的模型設定和目前載入的不同，請先重新執行 ④。")

    def make_progress(progress, head):
        def on_progress(value, desc):
            try:
                v = min(max(float(value), 0.0), 1.0) if value is not None else None
            except (TypeError, ValueError):
                v = None
            progress(v, desc=f"{head}｜{G['progress_label'](desc)}")
        return on_progress

    def read_text(path):
        try:
            return Path(path).read_text(encoding="utf-8-sig")
        except OSError:
            return ""

    def status_text(head, rows, notes=()):
        lines = [head]
        if rows:
            lines += ["", "本次結果："]
            lines += [f"  {r['song']}｜seed {r['seed']}｜{r['duration_sec']} 秒｜LRC {r['lrc']}｜{r['status']}" for r in rows]
        if notes:
            lines += [""] + list(notes)
        return "\n".join(lines)

    def timesig_value(value):
        try:
            return G["normalize_timesig"](value) or "auto"
        except ValueError:
            return "auto"

    def do_plan(*values, progress=gr.Progress()):
        v = dict(zip(KEYS, values))
        apply_settings(v)
        cancel.clear()
        t0 = time.time()
        try:
            plan = G["plan_structure"](on_progress=make_progress(progress, "先讓 LM 規劃曲式"), echo=False)
        except Exception as e:
            traceback.print_exc()
            raise gr.Error(f"規劃失敗：{e}"[:600]) from None
        if plan is None:
            return gr.update(), gr.update(), gr.update(), gr.update(), "⏹️ 已停止，曲式沒有變更。", gr.update()
        bpm = int(plan["bpm"]) if plan.get("bpm") else as_int(v["bpm"], "BPM", 0)
        duration = int(round(float(plan["duration"]))) if plan.get("duration") else as_int(v["duration"], "長度", -1)
        keyscale = plan.get("keyscale") or v["keyscale"]
        timesig = timesig_value(plan.get("timesignature")) if plan.get("timesignature") else v["timesig"]
        lm_caption = str(plan.get("caption") or "").strip()
        msg = (f"✅ LM 規劃完成（{time.time() - t0:.0f} 秒）：BPM {bpm or '—'}｜調性 {keyscale or '—'}｜"
               f"拍號 {G['timesig_label'](timesig_value(timesig))}｜長度 {duration if duration > 0 else '—'} 秒"
               + (f"｜語言 {plan['language']}" if plan.get("language") else "")
               + "\n曲式已填進右邊的欄位，可以直接修改再按「生成歌曲」。"
               + (f"\n\nLM 改寫的曲風描述（想用就按「套用 LM 改寫的曲風描述」）：\n{lm_caption}" if lm_caption else ""))
        return bpm, keyscale, timesig, duration, msg, lm_caption

    def do_generate(project, name_prefix, *values, progress=gr.Progress()):
        v = dict(zip(KEYS, values))
        apply_settings(v)
        cancel.clear()
        project = re.sub(r'[\\/:*?"<>|]+', "_", str(project or "").strip()) or "我的歌曲"
        project_dir = output_root / project
        project_dir.mkdir(parents=True, exist_ok=True)
        prefix = re.sub(r'[\\/:*?"<>|\s]+', "_", str(name_prefix or "").strip()) or str(G.get("FILE_PREFIX", "song_v"))
        digits = int(G.get("VERSION_DIGITS", 2))
        batch = int(G["BATCH_COUNT"])
        start = G["next_version_number"](project_dir, prefix)
        rows, notes, last_audio, last_lrc, stopped = [], [], None, "", False
        for i in range(batch):
            if cancel.is_set():
                stopped = True
                break
            song_name = f"{prefix}{start + i:0{digits}d}"
            if (project_dir / song_name).exists():
                raise gr.Error(f"{project_dir / song_name} 已存在，停止以免覆蓋。")
            seed = G["pick_seed"](i)
            head = f"[{i + 1}/{batch}] {song_name}（seed {seed}）"
            yield status_text(f"🎵 {head} 生成中…", rows, notes), last_audio, last_lrc
            try:
                out = G["generate_song"](project_dir, song_name, seed, on_progress=make_progress(progress, head), echo=False)
            except Exception as e:
                traceback.print_exc()
                rows.append({"song": song_name, "seed": seed, "duration_sec": "", "lrc": "", "mp3_ok": False,
                             "folder": "", "status": f"失敗：{e!r}"[:120]})
                yield status_text(f"❌ {head} 失敗，繼續下一首。", rows, notes), last_audio, last_lrc
                if not ACE.alive():
                    notes.append("❌ 背景模型已停止，請重新執行 ④ 後再開一次介面。")
                    break
                continue
            if out is None:
                stopped = True
                break
            row = out["row"]
            rows.append(row)
            notes += out["notes"]
            if row["status"] != "ok":
                yield status_text(f"❌ {head} 失敗，繼續下一首。", rows, notes), last_audio, last_lrc
                continue
            last_audio = out["audio"]
            if G["MAKE_LRC"] and not G["is_instrumental"]():
                last_lrc = out["lrc_text"] or (LRC_EMPTY if out["row"]["lrc"].startswith("0/") else
                                               "（LRC 產生失敗，詳見歌曲資料夾裡的 lrc_log.txt）")
            else:
                last_lrc = ""
            yield status_text(f"✅ {head} 完成，{row['duration_sec']} 秒。", rows, notes), last_audio, last_lrc
        if rows:
            G["write_batch_csv"](project_dir, rows)
        ok = sum(1 for r in rows if r["status"] == "ok")
        head = (f"⏹️ 已停止。完成 {ok} 首。" if stopped else f"🎉 全部完成：成功 {ok} 首，失敗 {len(rows) - ok} 首。")
        yield status_text(head + f"\n資料夾：{project_dir}", rows, notes), last_audio, last_lrc

    def do_stop():
        cancel.set()
        ACE.cancel()
        return "⏹️ 已送出停止要求，會在目前這個步驟的下一個檢查點停下…"

    # ── 歌曲瀏覽 ──
    def list_songs():
        found = []
        if output_root.is_dir():
            for proj in output_root.iterdir():
                if not proj.is_dir():
                    continue
                for song in proj.iterdir():
                    if song.is_dir() and (song / f"{song.name}.flac").exists():
                        found.append((song.stat().st_mtime, f"{proj.name}/{song.name}"))
        return [name for _, name in sorted(found, reverse=True)[:300]]

    def song_info(folder):
        try:
            return json.loads((folder / "params.json").read_text(encoding="utf-8"))
        except (OSError, ValueError):
            return {}

    def structure_of(info):
        """這首歌實際用的曲式：指定的優先，沒指定就用 LM 決定的。"""
        req, meta = info.get("request") or {}, info.get("lm_metadata") or {}
        bpm = req.get("bpm") or meta.get("bpm") or 0
        keyscale = req.get("keyscale") or meta.get("keyscale") or ""
        timesig = req.get("timesignature") or meta.get("timesignature") or ""
        duration = info.get("duration_sec") or meta.get("duration") or -1
        try:
            bpm, duration = int(float(bpm)), int(round(float(duration)))
        except (TypeError, ValueError):
            bpm, duration = 0, -1
        return bpm, str(keyscale), timesig_value(timesig), duration

    def show_song(rel):
        if not rel:
            return None, "", "", ""
        folder = (output_root / rel).resolve()
        if output_root.resolve() not in folder.parents or not folder.is_dir():
            return None, "", "找不到這個資料夾。", ""
        name = folder.name
        audio = next((p for p in (folder / f"{name}.mp3", folder / f"{name}.flac") if p.exists()), None)
        lrc_file = folder / f"{name}.lrc"
        lrc_text = (read_text(lrc_file).strip() or LRC_EMPTY) if lrc_file.exists() else "（這首沒有 LRC）"
        info = song_info(folder)
        summary = ""
        if info:
            bpm, keyscale, timesig, duration = structure_of(info)
            summary = (f"**曲式**：BPM {bpm or '—'}｜調性 {keyscale or '—'}｜拍號 {G['timesig_label'](timesig)}｜"
                       f"長度 {duration if duration > 0 else '—'} 秒｜seed {info.get('seed', '—')}｜{info.get('dit_model', '')}")
        return (str(audio) if audio else None, lrc_text, read_text(folder / "參數.txt") or "（這首沒有參數紀錄）", summary)

    def refresh_songs():
        items = list_songs()
        first = items[0] if items else None
        return (gr.update(choices=items, value=first),) + show_song(first)

    def use_structure(rel, with_seed):
        info = song_info(output_root / rel) if rel else {}
        if not info:
            raise gr.Error("這首沒有 params.json，無法帶入。")
        bpm, keyscale, timesig, duration = structure_of(info)
        gr.Info("曲式" + ("和 Seed " if with_seed else "") + "已帶到「生成」分頁。")
        if with_seed and info.get("seed") is not None:
            return bpm, keyscale, timesig, duration, "fixed", int(info["seed"])
        return bpm, keyscale, timesig, duration, gr.update(), gr.update()

    d = lambda name, default: G.get(name, default)
    model_label = f"{ACE.sig[0]} + {ACE.sig[1]}" if ACE.sig else ""
    with gr.Blocks(title="ACE-Step 歌曲生成") as demo:
        gr.Markdown(f"## 🎵 ACE-Step 1.5 歌曲生成\n模型：`{model_label}`｜歌曲存到 `{output_root}`")
        planned_caption = gr.State("")
        with gr.Tab("生成"):
            with gr.Row():
                with gr.Column(scale=5):
                    with gr.Row():
                        project = gr.Textbox(value=d("PROJECT_NAME", "我的歌曲"), label="專案名稱（歌曲會放在這個資料夾）")
                        name_prefix = gr.Textbox(value=d("FILE_PREFIX", "song_v"),
                                                 label="歌曲名稱（後面會自動加編號，例如 song_v01；可以用中文）")
                    caption = gr.Textbox(value=d("CAPTION", ""), lines=6,
                                         label="曲風描述 CAPTION（曲風、樂器、人聲、情緒…英文效果最穩，最多 512 字）")
                    lyrics = gr.Textbox(value=d("LYRICS", ""), lines=14,
                                        label="歌詞 LYRICS（[Verse]、[Chorus] 等段落標籤可保留；留空 = 純音樂）")
                    with gr.Row():
                        language = gr.Dropdown(LANG_CHOICES, value=d("VOCAL_LANGUAGE", "zh"), label="歌詞語言（unknown = 交給 LM）")
                        instrumental = gr.Checkbox(value=bool(d("INSTRUMENTAL", False)), label="純音樂（忽略歌詞）")
                        thinking = gr.Checkbox(value=bool(d("THINKING", True)), label="Thinking（LM 先規劃結構，建議開）")
                        batch = gr.Slider(1, 20, value=int(d("BATCH_COUNT", 1)), step=1, label="批量數")
                    with gr.Row():
                        seed_mode = gr.Dropdown(["random", "fixed", "sequence", "list"], value=d("SEED_MODE", "random"),
                                                label="Seed 模式")
                        fixed_seed = gr.Number(value=int(d("FIXED_SEED", 123456)), precision=0, label="固定 Seed（fixed／sequence 用）")
                        seed_list = gr.Textbox(value=d("SEED_LIST_TEXT", ""), label="Seed 清單（list 用，逗號分隔）")
                    with gr.Accordion("進階設定（不確定就保持預設）", open=False):
                        gr.Markdown(f"**DiT 音訊生成**（{model_label}）")
                        with gr.Row():
                            steps = gr.Number(value=int(d("INFERENCE_STEPS", 0)), precision=0,
                                              label="步數（0 = 預設：xl-sft 50｜xl-turbo 8）")
                            guidance = gr.Number(value=float(d("GUIDANCE_SCALE", 7.0)), label="CFG（只對 xl-sft 有效，1～15）")
                            shift = gr.Number(value=float(d("SHIFT", 3.0)), label="Shift（1～5）")
                            infer_method = gr.Dropdown(["ode", "sde"], value=d("INFER_METHOD", "ode"),
                                                       label="取樣方式（ode 固定｜sde 隨機性高）")
                        gr.Markdown("**LM 規劃**")
                        with gr.Row():
                            lm_temp = gr.Number(value=float(d("LM_TEMPERATURE", 0.85)), label="Temperature（0～2）")
                            lm_cfg = gr.Number(value=float(d("LM_CFG_SCALE", 2.0)), label="CFG（1～3）")
                            lm_top_k = gr.Number(value=int(d("LM_TOP_K", 0)), precision=0, label="Top K（0 = 不限）")
                            lm_top_p = gr.Number(value=float(d("LM_TOP_P", 0.9)), label="Top P（1.0 = 不限）")
                        with gr.Row():
                            lm_neg = gr.Textbox(value=d("LM_NEGATIVE_PROMPT", "NO USER INPUT"), label="LM Negative Prompt")
                            cot_caption = gr.Checkbox(value=bool(d("USE_COT_CAPTION", True)),
                                                      label="讓 LM 改寫、補充曲風描述（官方預設開）")
                    with gr.Accordion("LRC 歌詞時間軸與輸出", open=False):
                        with gr.Row():
                            make_lrc = gr.Checkbox(value=bool(d("MAKE_LRC", True)), label="產生 LRC")
                            lrc_method = gr.Radio(["acestep", "whisper"], value=d("LRC_METHOD", "acestep"),
                                                  label="LRC 方式（acestep 內建、快｜whisper 和 YuE2 版相同）")
                        gr.Markdown("以下只有 **whisper** 會用到")
                        with gr.Row():
                            lrc_lang = gr.Dropdown(["zh", "en", "ja"], value=d("LRC_LANGUAGE", "zh"), label="歌詞語言")
                            lrc_model = gr.Dropdown(["small", "medium", "large-v3"], value=d("LRC_WHISPER_MODEL", "medium"),
                                                    label="Whisper 模型", allow_custom_value=True)
                            lrc_device = gr.Dropdown(["cpu", "cuda"], value=d("LRC_DEVICE", "cpu"), label="LRC 裝置")
                        with gr.Row():
                            lrc_no_vad = gr.Checkbox(value=bool(d("LRC_NO_VAD", True)), label="不使用 VAD（慢歌、長音多建議勾選）")
                            lrc_interp = gr.Checkbox(value=bool(d("LRC_INTERPOLATE", False)), label="對不上的行用推測時間補齊")
                            lrc_separate = gr.Checkbox(value=bool(d("LRC_SEPARATE_VOCALS", True)),
                                                       label="先分離人聲再對歌詞（準確很多；第一次多花約 1 分鐘）")
                        with gr.Row():
                            make_mp3 = gr.Checkbox(value=bool(d("MAKE_MP3", True)), label="另存 MP3（FLAC 母檔一定保留）")
                            mp3_bitrate = gr.Dropdown(["192k", "256k", "320k"], value=d("MP3_BITRATE", "320k"), label="MP3 位元率")
                    with gr.Row():
                        go = gr.Button("生成歌曲", variant="primary", scale=3)
                        stop = gr.Button("停止", variant="stop", scale=1)
                with gr.Column(scale=4):
                    status = gr.Textbox(label="狀態", lines=10, interactive=False)
                    audio = gr.Audio(label="最新歌曲", type="filepath", interactive=False)
                    lrc_out = gr.Textbox(label="LRC 歌詞時間軸", lines=8, interactive=False)
                    gr.Markdown("### 曲式\n留空／自動 = 每首都交給 LM 決定。想先看 LM 怎麼規劃，就按「先讓 LM 規劃曲式」，改好再按「生成歌曲」。")
                    with gr.Row():
                        bpm = gr.Number(value=int(d("BPM", 0) or 0), precision=0, label="BPM（0 = 自動；30～300）")
                        keyscale = gr.Textbox(value=d("KEYSCALE", ""), label="調性（例：A minor；留空 = 自動）")
                    with gr.Row():
                        timesig = gr.Dropdown(TIMESIG_CHOICES, value=timesig_value(d("TIMESIGNATURE", "auto")), label="拍號")
                        duration = gr.Number(value=int(d("DURATION_SEC", -1)), precision=0, label="長度（秒；-1 = 自動；10～600）")
                    with gr.Row():
                        plan_btn = gr.Button("先讓 LM 規劃曲式（不生成歌曲）")
                        clear_btn = gr.Button("全部改回自動")
                    use_caption = gr.Button("套用 LM 改寫的曲風描述")
        with gr.Tab("歌曲瀏覽"):
            with gr.Row():
                songs = gr.Dropdown([], value=None, label="歌曲（新的在前）", scale=5)
                songs_refresh = gr.Button("重新整理", scale=1)
            with gr.Row():
                with gr.Column():
                    h_audio = gr.Audio(label="播放", type="filepath", interactive=False)
                    h_lrc = gr.Textbox(label="LRC 歌詞時間軸", lines=10, interactive=False)
                with gr.Column():
                    h_summary = gr.Markdown("")
                    h_params = gr.Textbox(label="生成參數", lines=14, interactive=False)
                    with gr.Row():
                        h_use = gr.Button("把這首的曲式帶到「生成」分頁")
                        h_use_seed = gr.Button("曲式＋Seed 一起帶入（重現這首）")

        inputs = [caption, lyrics, instrumental, language, duration, thinking,
                  batch, seed_mode, fixed_seed, seed_list,
                  bpm, keyscale, timesig,
                  steps, guidance, shift, infer_method,
                  lm_temp, lm_cfg, lm_top_k, lm_top_p, lm_neg, cot_caption,
                  make_lrc, lrc_method, lrc_lang, lrc_no_vad, lrc_interp, lrc_model, lrc_device,
                  lrc_separate, make_mp3, mp3_bitrate]
        history_out = [songs, h_audio, h_lrc, h_params, h_summary]
        go.click(do_generate, [project, name_prefix] + inputs, [status, audio, lrc_out],
                 concurrency_id="acestep", concurrency_limit=1).then(refresh_songs, None, history_out)
        plan_btn.click(do_plan, inputs, [bpm, keyscale, timesig, duration, status, planned_caption],
                       concurrency_id="acestep", concurrency_limit=1)
        stop.click(do_stop, None, status, queue=False)
        clear_btn.click(lambda: (0, "", "auto", -1), None, [bpm, keyscale, timesig, duration], queue=False)

        def apply_caption(text):
            if not str(text or "").strip():
                raise gr.Error("還沒有 LM 改寫的曲風描述，請先按「先讓 LM 規劃曲式」。")
            return text

        use_caption.click(apply_caption, planned_caption, caption)
        songs_refresh.click(refresh_songs, None, history_out)
        songs.input(show_song, songs, [h_audio, h_lrc, h_params, h_summary])
        h_use.click(lambda rel: use_structure(rel, False), songs, [bpm, keyscale, timesig, duration, seed_mode, fixed_seed])
        h_use_seed.click(lambda rel: use_structure(rel, True), songs, [bpm, keyscale, timesig, duration, seed_mode, fixed_seed])
        demo.load(refresh_songs, None, history_out)

    # 重新執行這格時，先把上一次開的介面關掉。
    old = G.get("ACESTEP_GUI_DEMO")
    if old is not None:
        try:
            old.close()
        except Exception:
            pass
    G["ACESTEP_GUI_DEMO"] = demo
    demo.queue()
    kw = dict(prevent_thread_lock=True, show_error=True, allowed_paths=[str(output_root)])
    kw.update(G.get("_ACESTEP_GUI_LAUNCH_OVERRIDES", {}))
    url = None
    if OPEN_IN_NEW_TAB:
        user, pw = "acestep", (GUI_PASSWORD.strip() or secrets.token_urlsafe(9))
        for attempt in range(2):
            try:
                _, _, url = demo.launch(share=True, auth=(user, pw), inline=False, quiet=True, **kw)
            except Exception as e:
                url = None
                print(f"（建立臨時網址時發生錯誤：{e}）")
            if url:
                break
            try:
                demo.close()      # 沒拿到網址：先關掉，重試或改用內嵌模式
            except Exception:
                pass
            if attempt == 0:
                print(" 第一次沒有拿到臨時網址，再試一次……")
        print("=" * 72)
        if url:
            print(f" 圖形介面網址：{url}\n 帳號：{user}\n 密碼：{pw}")
            print(" 這個網址是公開的，只在這次執行階段有效；請不要分享帳號密碼。")
        else:
            print(" 沒有成功產生臨時網址，自動改用內嵌模式：面板會直接顯示在這格下方，不需要密碼。")
        print("=" * 72)
    if not url:
        # 內嵌模式：告訴 Gradio 它在 Colab 代理後面的真正網址，音訊才載得出來。
        try:
            import socket
            from google.colab.output import eval_js
            port = 7860
            while port < 7900:
                with socket.socket() as s:
                    if s.connect_ex(("127.0.0.1", port)) != 0:
                        break
                port += 1
            root = str(eval_js(f"google.colab.kernel.proxyPort({port})")).rstrip("/")
            if root.startswith("http"):
                kw.setdefault("server_port", port)
                kw.setdefault("root_path", root)
        except Exception as e:
            print(f"（無法取得 Colab 代理網址，音訊預覽可能載不出來：{e}）")
        demo.launch(share=False, inline=True, height=1400, **kw)
    print("介面啟動後這格就算執行完了，介面會在背景持續運作。Colab 分頁請保持開啟；用完記得中斷執行階段。")


if USE_GUI:
    _acestep_gui_main()
else:
    print("⏭️ 未勾選 USE_GUI，請使用下面的 ⑤ 表單批量生成。")

## ⑤ 生成

每首歌一個資料夾：FLAC（母檔）、MP3、LRC、歌詞、`參數.txt`（含 LM 決定的曲式與改寫後的曲風描述）和 `params.json`。批次結束會另存一份摘要 CSV。

In [ ]:
# @title ⑤ 🎲 批量生成 + LRC
if globals().get("ACESTEP_GUI_ACTIVE"):
    print("⏭️ 已開啟圖形介面（⑤G），略過表單批量生成。想用表單批量，請取消勾選 ⑤G 的 USE_GUI，重新執行 ⑤G 再執行這格。")
else:
    if not ACE.alive():
        raise RuntimeError("請先執行 ④ 載入 ACE-Step。")
    if ACE.sig != (DIT_MODEL, LM_MODEL):
        raise RuntimeError("③ 的模型設定和目前載入的不同，請先重新執行 ④。")
    build_job(0)        # 先檢查設定，有錯會在這裡停下

    start_version = next_version_number(PROJECT_DIR, FILE_PREFIX)
    batch_rows, generated_dirs, batch_notes = [], [], []
    print(f"生成 {BATCH_COUNT} 首，從 {FILE_PREFIX}{start_version:0{VERSION_DIGITS}d} 開始 → {PROJECT_DIR}")

    for i in range(BATCH_COUNT):
        song_name = f"{FILE_PREFIX}{start_version + i:0{VERSION_DIGITS}d}"
        seed = pick_seed(i)
        print("\n" + "=" * 70 + f"\n🎵 [{i + 1}/{BATCH_COUNT}] {song_name}｜seed {seed}")
        try:
            out = generate_song(PROJECT_DIR, song_name, seed)
        except Exception as e:
            print("❌ 失敗：", repr(e))
            traceback.print_exc()
            batch_rows.append({"song": song_name, "seed": seed, "duration_sec": "", "lrc": "", "mp3_ok": False,
                               "folder": "", "status": f"失敗：{e!r}"[:120]})
            if not ACE.alive():
                print("❌ 背景模型已停止，請重新執行 ④。")
                break
            continue
        if out is None:
            print("⏹️ 已停止")
            break
        row = out["row"]
        batch_rows.append(row)
        batch_notes += out["notes"]
        for note in out["notes"]:
            print(note)
        if row["status"] != "ok":
            print(f"❌ {row['status']}（詳見 {row['folder']}/ERROR.txt）")
            continue
        generated_dirs.append(Path(row["folder"]))
        meta = out.get("meta") or {}
        lm_info = "｜".join(f"{label} {meta[k]}" for k, label in
                            (("bpm", "BPM"), ("keyscale", "調性"), ("duration", "長度")) if meta.get(k) not in (None, ""))
        print(f"⏱️ 長度 {row['duration_sec']} 秒｜LRC {row['lrc']}" + (f"｜LM：{lm_info}" if lm_info else ""))

    if batch_rows:
        summary = write_batch_csv(PROJECT_DIR, batch_rows)
        print("\n" + "=" * 70 + "\n🎉 完成｜摘要：", summary)
    for r in batch_rows:
        print(f"- {r['song']} | seed={r['seed']} | {r['duration_sec']} 秒 | LRC={r['lrc']} | {r['status']}")

In [ ]:
# @title ⑤B 🛠️ 補做 LRC（勾選才執行，不會重新生成歌曲）
# @markdown 用 whisper（Demucs + Whisper + lyric-align）替專案裡還沒有 LRC 的歌補做；設定沿用 ③ 的 whisper 選項。
RUN_LRC_REPAIR = False  # @param {type:"boolean"}
# @markdown 連已經有 LRC 的歌也重做（例如想把內建 LRC 換成 whisper 版本）
REDO_EXISTING_LRC = False  # @param {type:"boolean"}

if not RUN_LRC_REPAIR:
    print("⏭️ 未勾選，略過")
else:
    pattern = re.compile(rf"^{re.escape(FILE_PREFIX)}\d+$")
    for folder in [p for p in sorted(PROJECT_DIR.iterdir()) if p.is_dir() and pattern.match(p.name)]:
        name = folder.name
        lrc_path = folder / f"{name}.lrc"
        if lrc_path.exists() and not REDO_EXISTING_LRC:
            continue
        audio = folder / f"{name}.flac"
        lyrics = folder / f"{name}_lyrics.txt"
        if not audio.exists() or not lyrics.exists():
            print(f"⚠️ {name} 缺少音訊或歌詞檔，略過")
            continue
        if not lyrics.read_text(encoding="utf-8").strip():
            print(f"⏭️ {name} 是純音樂，略過")
            continue
        print("\n🛠️", name)
        if lrc_path.exists():
            shutil.copy2(lrc_path, folder / f"{name}.acestep.lrc")      # 保留原本的版本
        ok, log = run_lrc(audio, lyrics, lrc_path, folder / f"{name}_segments.json")
        (folder / "lrc_log.txt").write_text(log, encoding="utf-8")
        print(log)
        _lrc = LAST_LRC_INFO or {}
        _hit = f"對上 {_lrc['matched']}/{_lrc['total']} 行" if _lrc.get("total") else ""
        print(f"✅ 完成 {_hit}" if ok else f"❌ 失敗 {_hit}，詳見 lrc_log.txt")

In [ ]:
# @title ⑥ 🎧 預覽
from IPython.display import Audio, display

PREVIEW_LIMIT = 5  # @param {type:"slider", min:1, max:20, step:1}

shown = 0
for folder in globals().get("generated_dirs", [])[:PREVIEW_LIMIT]:
    f = folder / f"{folder.name}.mp3"
    f = f if f.exists() else folder / f"{folder.name}.flac"
    if f.exists():
        print("🎵", folder.name)
        display(Audio(str(f)))
        shown += 1
if shown == 0:
    print("本批沒有可預覽的歌曲。")

In [ ]:
# @title ⑦ 🧹 釋放 GPU 記憶體（勾選才執行）
# @markdown 會關掉背景模型；釋放後，下次生成前需重新執行 ④
RELEASE_GPU = False  # @param {type:"boolean"}

if not RELEASE_GPU:
    print("⏭️ 未勾選，保留已載入的 ACE-Step")
elif not ACE.alive():
    print("背景模型本來就沒有在跑")
else:
    ACE.stop()
    print("✅ 已釋放")

In [ ]:
# @title ⑧ 💾 備份到 Google Drive（自動，已備份會略過）
# 程式碼打包成 tar；模型每個資料夾各自備份，之後換用其他模型時只會補備份新的部分。
# @markdown 重新備份（例如官方更新了模型）
FORCE_REBACKUP = False  # @param {type:"boolean"}

if not drive_ready():
    print("⏭️ 未啟用 Drive 備份（① 的 USE_DRIVE_BACKUP）")
else:
    BACKUP_DIR.mkdir(parents=True, exist_ok=True)

    # 程式碼（不含 venv 和模型）
    if "acestep_code" in ACE_RESTORED:
        print("✅ acestep_code：本次即由備份還原，略過")
    elif backup_done("acestep_code") and not FORCE_REBACKUP:
        print("✅ acestep_code：已備份")
    elif (ACE_REPO / "pyproject.toml").exists():
        print("💾 備份 acestep_code ……")
        backup_code_tar("acestep_code", ACE_REPO, exclude={".venv", "checkpoints"})
        print("✅ acestep_code：完成")

    # 模型
    if ACE.sig is None:
        print("⏭️ 模型尚未載入成功（請先執行 ④），略過模型備份")
    else:
        for name in needed_models():
            src = CKPT_DIR / name
            key = f"ckpt_{name}"
            if not src.is_dir():
                continue
            if key in ACE_RESTORED:
                print(f"✅ {key}：本次即由備份還原，略過")
            elif backup_done(key) and not FORCE_REBACKUP:
                print(f"✅ {key}：已備份")
            else:
                print(f"💾 備份 {key} ……")
                try:
                    backup_dirs(key, {name: src})
                    print(f"✅ 完成（{dir_size_gb(BACKUP_DIR / key):.1f} GB）")
                except Exception as e:
                    print(f"⚠️ 備份失敗：{e!r}（Drive 空間不足時常見）")

    print(f"\n備份位置：{BACKUP_DIR}")
    print("ℹ️ Drive 會在背景上傳，剛備份完時請等幾分鐘再中斷 Runtime。")